The default option to estimate production data, whenever another data source is not available is the exiobase database. To read exiobase, we rely on the pymrio package (https://pymrio.readthedocs.io).

In [1]:
import pandas as pd
import pymrio

Select the exiobase version you want to use to estimate production data.

In [2]:
io = pymrio.parse_exiobase3('C://Users/11max/Desktop/Work/Databases/EXIOBASE/monetary/exiobase3.8.2/IOT_2022_pxp/')
io.calc_all()

Simply loop through regions and sectors of exiobase and for every combination, estimate how much of the domestic production is used domestically vs is exported to other countries.

In [11]:
domestic_production_estimates = pd.Series()

for region in io.get_regions():
    for product in io.get_sectors():
        domestic_consumption = io.Z.loc(axis=0)[region, product].loc[region].sum() + io.Y.loc(axis=0)[region, product].loc[region].sum()
        total_production = (domestic_consumption + (io.Z.loc(axis=0)[region, product].sum() + io.Y.loc(axis=0)[region, product].sum()) - (
            io.Z.loc(axis=0)[region, product].loc[region].sum() + io.Y.loc(axis=0)[region, product].loc[region].sum()))
        if total_production != 0:
            domestic_production = domestic_consumption / total_production * 100
        else:
            domestic_production = 0
        domestic_production_estimates = pd.concat([domestic_production_estimates, pd.Series(domestic_production, index=[(region, product)])])

domestic_production_estimates.index = pd.MultiIndex.from_tuples(domestic_production_estimates.index)

C:\Users\11max\AppData\Local\Temp\ipykernel_9156\1979922882.py:12: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  domestic_production_estimates = pd.concat([domestic_production_estimates, pd.Series(domestic_production, index=[(region, product)])])


There are cases where the estimation from exiobase results in extremely low values (close to zero) or extremely high values (close to one). In both cases, this most likely results from very rare cases within the economy of the region, or simply to artefacts due to modeling approximations. To avoid further complications, we fixed values lower than 0.1% to 0% and values higher than 99.95 to 100%.

In [12]:
domestic_production_estimates.loc[domestic_production_estimates < 0.1] = 0
domestic_production_estimates.loc[domestic_production_estimates > 99.9] = 100

We then simply write the results in the SQL database. Note that here the code only picks one year for the exiobase data. You could also run this for multiple years and average the results. Realistically though, this would not change much as the estimation through exiobase is extremely coarse anyways.

In [ ]:
trade_conn = sqlite3.connect('C://Users/11max/PycharmProjects/Regioinvent/trade_data/v5/trade_data.db')

domestic_production_estimates.to_sql('Domestic use ratios exiobase', trade_conn)